# Medical Waste Data Cleaning and Preprocessing
Issue #3 — MED-WASTE-SEGREGATION. This notebook preserves the raw input, creates a cleaned dataset, records special-value decisions, validates the result and flags (but does not delete) potential IQR outliers.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from preprocessing import load_raw, clean_dataframe, validate_dataframe, iqr_outlier_summary

raw_candidates = [ROOT / 'medical_waste.csv', ROOT / 'data' / 'raw' / 'medical_waste.csv']
RAW_PATH = next((p for p in raw_candidates if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError('Place the original medical_waste.csv in the repository root or data/raw/.')
OUTPUT_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load and inspect the raw data


In [ ]:
raw_df = load_raw(RAW_PATH)
print('Shape:', raw_df.shape)
display(raw_df.head())
print('\nRaw missing/special token counts:')
for token in ['INP', 'NIL', 'NA', '']:
    print(repr(token), (raw_df == token).sum().sum())


## 2. Clean values and preserve an audit trail


In [ ]:
cleaned_df, audit_df = clean_dataframe(raw_df)
display(cleaned_df.head())
print('Audit entries:', len(audit_df))
display(audit_df.head(20))


## 3. Missing-value profile


In [ ]:
missing_profile = cleaned_df.isna().sum().rename('missing_count').to_frame()
missing_profile['missing_percent'] = (missing_profile['missing_count'] / len(cleaned_df) * 100).round(2)
display(missing_profile[missing_profile.missing_count > 0].sort_values('missing_count', ascending=False))


## 4. Duplicate and schema validation


In [ ]:
validation_df = validate_dataframe(cleaned_df)
display(validation_df)
assert validation_df['passed'].all(), 'One or more validation checks failed.'


## 5. Potential outliers (review only)
No aggregated State/UT record is removed based only on an IQR flag.


In [ ]:
outlier_df = iqr_outlier_summary(cleaned_df)
display(outlier_df.sort_values('flagged_count', ascending=False))


## 6. Save outputs


In [ ]:
cleaned_path = OUTPUT_DIR / 'medical_waste_cleaned.csv'
audit_path = OUTPUT_DIR / 'special_value_audit.csv'
cleaned_df.to_csv(cleaned_path, index=False)
audit_df.to_csv(audit_path, index=False)
validation_df.to_csv(OUTPUT_DIR / 'validation_results.csv', index=False)
outlier_df.to_csv(OUTPUT_DIR / 'iqr_outlier_review.csv', index=False)
print('Saved:', cleaned_path)
print('Saved:', audit_path)


## 7. Summary
- Raw data remains unchanged.
- `NIL` is converted to zero based on the repository's source documentation.
- `INP`, `NA`, blanks and explanatory text in numeric columns remain missing and are documented in the audit.
- No rows are dropped; outliers are review flags only.
- No scaling/encoding is applied because this is an analytics-ready aggregate dataset.
